In [4]:
import pandas as pd
import numpy as np
from portfolio_optimizer.optimizer import StaticOptimizer
from portfolio_optimizer.constraints import create_style_constraints
from portfolio_optimizer.utils import port_perf, exp_utility

# Generate dummy data for two scenarios (e.g., Low Vol vs High Vol)
np.random.seed(42)
n_assets = 10
asset_names = [f'Asset_{i+1}' for i in range(n_assets)]
growth_assets = asset_names[:6] # First 6 assets are categorized as 'Growth'

# Scenario 1: Stable Market
mu1 = np.random.uniform(0.05, 0.12, n_assets)
A1 = np.random.rand(n_assets, n_assets)
sigma1 = np.dot(A1, A1.T) * 0.02

# Scenario 2: Volatile Market
mu2 = mu1 + np.random.uniform(-0.02, 0.04, n_assets)
A2 = np.random.rand(n_assets, n_assets)
sigma2 = np.dot(A2, A2.T) * 0.05 # Higher risk

rf_annual = 0.02
print(f"Toolkit ready. Analyzing {n_assets} assets under two market scenarios.")

Toolkit ready. Analyzing 10 assets under two market scenarios.


In [5]:
# (Growth_Min, Growth_Max, Label)
risk_profiles = [
    (0.25, 0.35, "Defensive (30/70)"),
    (0.65, 0.75, "Balanced (70/30)"),
    (0.85, 0.95, "Aggressive (90/10)")
]

opt1 = StaticOptimizer(mu1, sigma1)
opt2 = StaticOptimizer(mu2, sigma2)

bounds = tuple((0.0, 0.40) for _ in range(n_assets))
results_data = []

for opt, set_name in [(opt1, "Stable Market"), (opt2, "Volatile Market")]:
    for g_low, g_high, label in risk_profiles:
        # Generate style-specific constraints using the factory
        style_cons = create_style_constraints(asset_names, growth_assets, g_low, g_high)
        
        # Solve for Maximum Sharpe Ratio
        res = opt.maximize_sharpe(rf=rf_annual, bounds=bounds, constraints=style_cons)
        
        if res.success:
            w = np.clip(res.x, 0, 1)
            r, v = port_perf(w, opt.mu, opt.cov)
            s = (r - rf_annual) / (v + 1e-12)
            u = exp_utility(r, v, gamma=1.0)
            
            results_data.append({
                "Scenario": set_name, "Profile": label,
                "Return": r, "Volatility": v, "Sharpe": s, "Utility": u
            })

In [6]:
perf_df = pd.DataFrame(results_data).set_index(["Scenario", "Profile"])

# Use Pandas Styler for clean, professional output
styled_df = perf_df.style.format({
    'Return': '{:.2%}',
    'Volatility': '{:.2%}',
    'Sharpe': '{:.4f}',
    'Utility': '{:.4f}'
})

print("\nPerformance Analysis: Market Scenarios vs. Investor Profiles")
display(styled_df)


Performance Analysis: Market Scenarios vs. Investor Profiles
